## Creating research and writing agents using crewai 

In [1]:
from dotenv import dotenv_values
import os 
from openai import OpenAI

In [2]:
config = dotenv_values(".env")

# os.environ["OPENAI_API_KEY"] = config['OPENAI_API_KEY']
os.environ["GEMINI_API_KEY"] = config['GEMINI_API_KEY']
# os.environ["DEEPSEEK_API_KEY"] = config['DEEPSEEK_API_KEY']

In [3]:
# config['OPENAI_API_KEY']

In [4]:
client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"], 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

print("client initialized")

client initialized


In [5]:
os.environ["OPENAI_MODEL_NAME"] = 'gemini-3.1-flash-lite'

In [ ]:
# response = client.responses.create(
#   model="deepseek-chat",
#   input="write a haiku about ai",
#   store=True,
# )

response = client.chat.completions.create(
    model=os.environ["OPENAI_MODEL_NAME"],
    messages=[
        {"role": "user", "content": "write a haiku about ai"}
    ]
)

print(response.choices[0].message.content) 

In [6]:
# warning control
import warnings
warnings.filterwarnings('ignore')

In [7]:
from crewai import Agent, Task, Crew, LLM

In [8]:
llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"]
)

In [9]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory=  "You're working on planning a blog article "
                "about the topic: {topic}." 
                "You collect information that helps the audience learn something "
                "and make more informed decisions. "
                "Your work is the basis for the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
    verbose=False
)

In [10]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=False
)

In [11]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    llm=llm,
    verbose=False
)

In [12]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

In [13]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

In [14]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

In [15]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=False
)

In [16]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

In [18]:
from IPython.display import Markdown
Markdown(result.raw)

# The AI Navigator: A Pragmatic Guide to the Future of Work

The rapid evolution of technology has left many feeling caught in a perpetual "hype cycle." Every week brings news of a new model or a breakthrough that promises to change everything, often leaving professionals feeling more overwhelmed than empowered. It is easy to view these developments through a lens of apprehension; however, it is essential to recognize that we are not merely witnessing a fleeting trend, but rather a foundational shift in how we process information and solve complex problems.

The most significant challenge today is not that artificial intelligence will replace our roles, but that the anxiety surrounding it will prevent us from engaging with it meaningfully. By framing AI as a foundational shift—akin to the advent of the internet—we can stop treating it as a mysterious disruptor and start viewing it as an infrastructure for productivity. The goal of this guide is to move you from a state of intimidation to one of active, informed participation.

## The Evolution of Intelligence: From Chatbots to Agents

To navigate this landscape, it is helpful to distinguish between the two primary tiers of technology currently dominating the conversation: Generative AI and Agentic AI. Generative models, such as GPT-4o or Claude, excel at synthesizing information, creative drafting, and coding assistance. They function as powerful engines for content generation, operating primarily on a prompt-response basis that aids in daily administrative and creative tasks.

The industry is currently shifting toward Agentic AI—systems designed to execute multi-step workflows with increased autonomy. Unlike a standard chatbot that answers a direct question, an agent can be tasked with researching a topic, summarizing findings, and drafting a report, performing each step in sequence. While this transition represents a significant technical advancement, it is important to remember that these systems are still evolving and function most effectively as sophisticated assistants rather than independent decision-makers.

## Productivity over Panic: Reframing Your Workflow

The practical application of AI in the workplace often seems daunting because there is a tendency to focus on speculative future capabilities rather than current utility. Utilizing AI for daily tasks like document summarization, technical code refinement, or brainstorming creative directions can save significant amounts of cognitive labor. The objective reality is that these tools provide the most value when you maintain "human-in-the-loop" oversight, ensuring that your unique perspective guides the final output.

Treating an AI model as a highly capable, albeit fallible, assistant is an effective integration strategy. When you use AI to draft an outline or structure a complex report, you are not abdicating your responsibility; you are accelerating your output. By offloading the initial "blank page" phase to an AI, you gain the time to focus on high-level strategy, critical thinking, and the nuanced communication that remains a distinct human advantage.

## The Ethical Compass: Navigating Hallucinations and Bias

As we integrate these tools, maintaining a high standard for AI ethics is critical. Current large language models are prone to "hallucinations," where they may confidently present incorrect information as fact. Furthermore, because these models are trained on vast, aggregated datasets, they can reflect existing biases. These are not merely technical glitches, but inherent characteristics of the technology that require users to remain diligent and skeptical.

Developing strong information literacy is perhaps the most vital skill for the modern professional. Relying on an AI to do your thinking is a risk; using an AI to provide data that you then verify and synthesize is a strategy. As the EU AI Act and other global frameworks begin to standardize AI governance, the responsibility falls on the user to prioritize data privacy and verify all generated outputs before applying them in professional contexts.

## Preparing for a Human-Led Future

Preparing for the future of AI is less about mastering every technical nuance of machine learning and more about cultivating adaptability. The "human-in-the-loop" advantage ensures that while machines handle the heavy lifting of data processing, humans retain the ability to provide empathy, ethical judgment, and complex context—factors that AI currently cannot replicate.

Continuous learning serves as the best hedge against uncertainty. By experimenting with diverse tools—ranging from small language models for local processing to multimodal models for visual data analysis—you build the necessary skills for the next wave of technological change. Do not just watch the shift—lead it. 

**Don't just watch the shift—lead it. Sign up for our weekly 'AI Digest' newsletter to get one actionable prompt per week to transform your workflow.**

In [19]:
topic = "What power is based on Game of Thrones"
result = crew.kickoff(inputs={"topic": topic})

In [20]:
Markdown(result.raw)

# The Iron Throne: Analyzing the Real-World Power Dynamics of *Game of Thrones*

"Power resides where men believe it resides." This iconic line, whispered by Varys to Tyrion Lannister, serves as the cornerstone of the political philosophy within the world of *Game of Thrones*. While the series is a triumph of high-fantasy storytelling, it functions primarily as a masterclass in political science. By examining the rise and fall of various houses, we can identify three distinct types of power at play: coercive power, reward-based power, and the far more elusive influential power.

*Game of Thrones* remains a vital text for students of history and leadership because it strips away the veneer of fantasy to reveal the raw machinery of statecraft. Whether one prefers George R.R. Martin’s novels or the televised adaptations, the series offers a profound look at how political structures—from feudalism to emerging meritocracies—actually function when the stakes are life and death.

## The Currency of Control: Understanding Hard Power
At the center of Westerosi politics is the concept of "Hard Power," a term often associated with Joseph Nye’s academic framework. Tywin Lannister acts as the quintessential practitioner of this, famously asserting that gold is the bedrock of power. Tywin understood that armies, wealth, and control over resources are the tangible tools used to maintain a regime. When a leader relies on the threat of force or the promise of economic reward, they are utilizing coercive and reward-based power.

Historically, this reliance on tangible assets allowed House Lannister to project strength for decades. However, this strategy faced a critical flaw: it often ignored the concept of the "Social Contract." The failure of such regimes often stems from the inability to recognize that without the consent or the belief of the governed, hard power eventually leads to isolation. When the gold reserves are exhausted and military supremacy wanes, a leader built solely on transactional power is often left with little institutional support.

## The Information Game: Asymmetric Knowledge
While Tywin hoarded gold, characters like Littlefinger and Varys mastered the "Information Game." They operate on the principle of asymmetric information—possessing knowledge that their opponents lack. In the political landscape of Westeros, this is often depicted as being more lethal than a Valyrian steel blade. By manipulating the flow of intelligence, these characters were able to destabilize entire kingdoms without ever drawing a weapon in the traditional, military sense.

This demonstrates a shift from physical dominance to strategic manipulation. In a modern context, this parallels how data and narrative control dictate contemporary geopolitics and corporate environments. Littlefinger, in particular, represents a specific archetype in power structures: the agent who navigates instability to advance personal interests, illustrating that in a system of shifting loyalties, information acts as a currency that does not depreciate as easily as physical goods.

## The Legitimacy Crisis and the Cost of Governance
The series also highlights the ongoing tension between "Divine Right" and meritocracy. Characters such as Daenerys Targaryen and Stannis Baratheon often justified their claims through bloodlines and ideological purity. Yet, the narrative arc suggests that legitimacy is rarely inherited; it is built through consensus and effective governance. Those who relied solely on their abstract claim to the throne often found themselves trapped in a paradox: the higher they climbed, the more isolated they became from the populace they aimed to rule.

The Iron Throne itself serves as a potent symbol for the burden of leadership. It is not a seat of comfort, but a jagged, dangerous construct that requires constant vigilance. Characters who survive these trials often demonstrate that power is fluid rather than static. While the "Game" is frequently viewed as a quest for total control, the most successful players eventually realize that the true cost of governance involves the sacrifice of personal agency and the necessity of constant political negotiation.

## Real-World Applications: Power in Your Community
One does not need a dragon or an army to observe these dynamics in action. These same power structures manifest in the workplace and local government daily. The "Tywins" of the corporate world rely on salary and status to maintain order, while the "Littlefingers" navigate internal politics through intelligence and influence. Recognizing these patterns is the first step toward effective leadership and, perhaps more importantly, navigating the complex political environments inherent in any organization.

The danger, as seen in the arc of Cersei Lannister, lies in relying on a singular, brittle source of power. When a leader purges opposition and relies solely on fear, they often lose the support of the very systems—such as legal frameworks and communal institutions—that keep a state functional. A sustainable power structure typically requires a balance of influence, consensus-building, and the ability to adapt to changing societal tides.

Power in *Game of Thrones* is never a final destination; it is a constant, shifting process of negotiation and sacrifice. Whether you identify with the pragmatic strategist or the ideological visionary, the series invites us to reflect on what we value in our own leaders. Which character’s approach to power do you admire most—the pragmatic strategist or the ideological visionary? Join the conversation in the comments below.